# **Reward Modeling with GPT-2**

Imagine you're working as a machine learning engineer at a healthcare technology company that wants to integrate advanced language models into its medical assistant platform. Your task is to evaluate and select the best large language model (LLM) that can accurately interpret patient queries, assist doctors with clinical documentation, and generate clear, empathetic, and medically sound responses.

However, simply deploying a powerful LLM isn't enough. A general-purpose model might give technically correct answers while being too cold in tone, overly verbose, or even potentially misleading in a medical context. This is where reward models become essential. By training a reward model, you can teach the LLM to prioritize the right behaviors — such as being concise, compassionate, and clinically accurate — ensuring its responses align with both medical best practices and the company's commitment to patient safety.

In this hands-on lab, you dive into the process of building and training a reward model using the Transformer Reinforcement Learner (TRL) library from Hugging Face. You'll learn how to configure the training environment, define meaningful reward signals that reflect what a good medical response looks like, and fine-tune the LLM to consistently produce high-quality, context-aware output. By the end, you'll have practical experience applying reward modeling to a real-world domain — giving you the tools to shape AI behavior in any high-stakes application.

### **Import Libraries**

In [110]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from datasets import load_dataset
from datasets import DatasetDict
from transformers import AutoTokenizer, GPT2ForSequenceClassification
from peft import LoraConfig, TaskType, get_peft_model

In [75]:
dataset_name = "Dahoas/synthetic-instruct-gptj-pairwise"

In [76]:
dataset = load_dataset(dataset_name)

In [77]:
dataset

DatasetDict({
    train: Dataset({
        features: ['prompt', 'chosen', 'rejected'],
        num_rows: 33143
    })
})

In [78]:
model = GPT2ForSequenceClassification.from_pretrained("gpt2")
tokenizer = AutoTokenizer.from_pretrained("gpt2")

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 3289.81it/s]
[transformers] GPT2ForSequenceClassification LOAD REPORT from: gpt2
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [79]:
# Add special tokens if necessary
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = model.config.eos_token_id

max_length = 1024

In [80]:
dataset["train"][0]

{'prompt': 'I was wondering if you could walk me through the process of setting up a hydroponic garden for herbs.',
 'chosen': "Sure! The process for setting up a hydroponic garden for herbs is relatively simple. First, you'll want to choose a space where you will set up your hydroponic system. You'll need to make sure the space is well-lit and has access to electricity and an adequate water supply. Next, you'll need to choose the type of hydroponic system you want to use. There are several types of hydroponic systems, so you'll need to decide which best suits your needs. Once you've chosen a system, you'll need to gather the supplies you'll need to assemble it. This includes things like pumps, growing trays, grow lights, and nutrients. Once you've assembled the system, you'll need to add your choice of herbs to the system. Lastly, you'll need to monitor and adjust the system as needed to ensure your herbs are getting the correct amount of light, water, and nutrients.",
 'rejected': 'H

**Need to filter the chosen and rejected responses with the prompt (which user's input). To do that we are going to use a Lamda Function with the relevant respose type**

In [81]:
get_res = lambda dataset, res_type: ["\n\nHuman: " + prompt + "\n\nAssistant: "+ response 
                                    for prompt, response in zip(dataset['train']['prompt'], dataset['train'][res_type])]

In [82]:
res_chosen = get_res(dataset, 'chosen') 
res_rejected = get_res(dataset, 'rejected')

In [83]:
for i in range (6):
    print ("response chosen:", res_chosen[i])



response chosen: 

Human: I was wondering if you could walk me through the process of setting up a hydroponic garden for herbs.

Assistant: Sure! The process for setting up a hydroponic garden for herbs is relatively simple. First, you'll want to choose a space where you will set up your hydroponic system. You'll need to make sure the space is well-lit and has access to electricity and an adequate water supply. Next, you'll need to choose the type of hydroponic system you want to use. There are several types of hydroponic systems, so you'll need to decide which best suits your needs. Once you've chosen a system, you'll need to gather the supplies you'll need to assemble it. This includes things like pumps, growing trays, grow lights, and nutrients. Once you've assembled the system, you'll need to add your choice of herbs to the system. Lastly, you'll need to monitor and adjust the system as needed to ensure your herbs are getting the correct amount of light, water, and nutrients.
respo

In [84]:
for i in range (6):
    print ("response rejected:", res_rejected[i])

response rejected: 

Human: I was wondering if you could walk me through the process of setting up a hydroponic garden for herbs.

Assistant: How do I store a bagels for eating at a later date?


 You can place the bagels in an airtight container and reheat them in the microwave.  Alternately, you can place the bagels in the microwave, cover them with foil, then heat them in the microwave for a short time.
response rejected: 

Human: What type of wine goes best with steak.

Assistant: Choose leaner cuts of steak or a burger dish, because the flavor of wine is the most versatile.  For example, cabernet sauvignon or pinot gallo works well with burritos, but the meat itself has to be grilled and fried.  
2.  Pour a good quality dry white wine that’s moderately young.  Chardonnay or far-lauter is also nice for steak.  
3.  Sauvignon Blanc is a nice, full-bodied white wine that works well with steak.
4.  Chenin Blanc is a grape that grows in northwest France, so it often tastes a bit strong

**It is more helpful when we have seperate columns to define prompt with chosen and prmopt with rejected.**

In [85]:
def add_seperate_res_column(mydataset):
    
    mydataset['prompt_chosen'] = "\n\nHuman: "+ mydataset['prompt'] + "\n\nAssistant: "+ mydataset['chosen']
    mydataset['prompt_rejected'] = "\n\nHuman" + mydataset['prompt'] + "\n\nAssistant: " + mydataset['rejected']
    
    return mydataset
    

In [86]:
dataset['train'] = dataset['train'].map(add_seperate_res_column)

In [87]:
dataset['train'][0]

{'prompt': 'I was wondering if you could walk me through the process of setting up a hydroponic garden for herbs.',
 'chosen': "Sure! The process for setting up a hydroponic garden for herbs is relatively simple. First, you'll want to choose a space where you will set up your hydroponic system. You'll need to make sure the space is well-lit and has access to electricity and an adequate water supply. Next, you'll need to choose the type of hydroponic system you want to use. There are several types of hydroponic systems, so you'll need to decide which best suits your needs. Once you've chosen a system, you'll need to gather the supplies you'll need to assemble it. This includes things like pumps, growing trays, grow lights, and nutrients. Once you've assembled the system, you'll need to add your choice of herbs to the system. Lastly, you'll need to monitor and adjust the system as needed to ensure your herbs are getting the correct amount of light, water, and nutrients.",
 'rejected': 'H

**Identifyting the maximum sample length in the dataset**

In [88]:
get_max_length = lambda data_list: max([len(sample) for sample in data_list])

In [89]:
max_len_chosen = get_max_length(res_chosen)
max_len_rejected = get_max_length(res_rejected)

print(f"max length of res_chosen:{max_len_chosen}")
print(f"max length of res_rejected:{max_len_rejected}")

max length of res_chosen:3167
max length of res_rejected:5011


**We need to filter the datset based on the model's max length (=1024). So if we get data that beyond the model's max length, the row data will be truncate and there have a chance to miss the important information that valuable to train the model.**

In [90]:
find_ids = lambda mydataset, max_len: [
    i for i, (chosen,rejected) in enumerate(zip(mydataset['prompt_chosen'], mydataset['prompt_rejected']))
    if len(chosen) < max_len or len(rejected)< max_len
]

In [91]:
selected_ids = find_ids(dataset['train'], max_length)


In [92]:
selected_ids[0:10]

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

In [93]:
dataset['train'] = dataset['train'].select(selected_ids)


In [94]:
dataset['train'][0]

{'prompt': 'I was wondering if you could walk me through the process of setting up a hydroponic garden for herbs.',
 'chosen': "Sure! The process for setting up a hydroponic garden for herbs is relatively simple. First, you'll want to choose a space where you will set up your hydroponic system. You'll need to make sure the space is well-lit and has access to electricity and an adequate water supply. Next, you'll need to choose the type of hydroponic system you want to use. There are several types of hydroponic systems, so you'll need to decide which best suits your needs. Once you've chosen a system, you'll need to gather the supplies you'll need to assemble it. This includes things like pumps, growing trays, grow lights, and nutrients. Once you've assembled the system, you'll need to add your choice of herbs to the system. Lastly, you'll need to monitor and adjust the system as needed to ensure your herbs are getting the correct amount of light, water, and nutrients.",
 'rejected': 'H

### **Data Tokenization Function**

The preprocess_function converts the prompt_chosen and prompt_rejected fields into tokens, which the RewardTrainer needs to function properly. Think of chosen as the "good answer" and rejected as the "bad answer."
By tokenizing both, the model can numerically compare what makes one response better than the other. Feeding the trainer both examples side by side is what allows it to learn the difference between a high-quality and a low-quality output — which is the whole point of reward-based training: teaching the model to consistently favor better responses.

In [95]:
def data_tokenization (mydataset):
    
    tokenized_chosen = tokenizer(mydataset['prompt_chosen'], 
                                max_length=max_length, padding='max_length', truncation=True)

    tokenized_rejected = tokenizer(mydataset['prompt_rejected'], max_length= max_length, 
                                padding = 'max_length', truncation=True)
    
    return {
        "input_ids_chosen": tokenized_chosen['input_ids'],
        "attention_mask_chosen": tokenized_chosen['attention_mask'],
        "input_ids_rejected": tokenized_rejected['input_ids'],
        "attention_mask_rejected": tokenized_rejected['attention_mask']
    }

#### Create a Dictionary to Access Chosen and Rejected Prompt for Model Validation

In [96]:
train_str = {"chosen": [data for data in dataset['train']['prompt_chosen']], "rejected": [data for data in dataset['train']['prompt_rejected']]}

Map the tokenized function with dataset:

In [117]:
dataset_tokenized:DatasetDict = {'train': {}}

In [122]:
dataset_tokenized = DatasetDict({
    "train": dataset['train'].map(data_tokenization, batched=True,
            remove_columns=['prompt',"chosen", "rejected",'prompt_chosen', 'prompt_rejected'])
})

In [123]:
dataset_tokenized

DatasetDict({
    train: Dataset({
        features: ['input_ids_chosen', 'attention_mask_chosen', 'input_ids_rejected', 'attention_mask_rejected'],
        num_rows: 33043
    })
})

Split the tokenized data set into train and test partitions

In [124]:
dataset_split = dataset_tokenized['train'].train_test_split(test_size=0.2, seed=42)

In [125]:
dataset_dict = DatasetDict({
    "train": dataset_split['train'],
    "test": dataset_split['test'],
})

In [126]:
dataset_dict

DatasetDict({
    train: Dataset({
        features: ['input_ids_chosen', 'attention_mask_chosen', 'input_ids_rejected', 'attention_mask_rejected'],
        num_rows: 26434
    })
    test: Dataset({
        features: ['input_ids_chosen', 'attention_mask_chosen', 'input_ids_rejected', 'attention_mask_rejected'],
        num_rows: 6609
    })
})